# Beam Search Algorithm Implementation

Beam Search is a heuristic search algorithm that explores a graph by expanding the most promising nodes in a limited set. Unlike Best-First Search or BFS which keep all generated nodes in memory, Beam Search only keeps the top $W$ nodes (where $W$ is the **Beam Width**) at each level of the search tree.

### Key Concepts:
- **Beam Width ($W$):** The maximum number of states kept at each level.
- **Evaluation Function ($f(n) = g(n) + h(n)$):** Used to score and rank candidate paths.
- **Trade-off:** Lower memory usage, but it is not guaranteed to find the optimal path (incomplete).


### Step 2: Implementation of Beam Search

The implementation tracks candidate paths. At each depth step, it expands all current paths, evaluates them using $f(n) = g(n) + h(n)$, and keeps only the top $W$ paths.


In [1]:
def beam_search(graph, heuristics, start, goal, beam_width):
    # Each path is stored as a tuple: (total_f_score, current_g_score, path_list)
    # Start path has g(start) = 0, f(start) = 0 + h(start)
    current_beams = [(heuristics[start], 0, [start])]

    print(f"Starting Beam Search with Beam Width W = {beam_width}\n")

    step = 0
    while current_beams:
        step += 1
        print(f"--- Step {step} ---")
        for f, g, path in current_beams:
            print(f"Active Path: {' -> '.join(path)} (f: {f}, g: {g})")

        # Check if any of the current beams reached the goal
        for _, cost, path in current_beams:
            if path[-1] == goal:
                print("\nGoal Reached!")
                return path, cost

        all_candidates = []

        # Expand all current paths
        for f_val, g_val, path in current_beams:
            last_node = path[-1]

            # If node has no neighbors or is the goal, we cannot expand it further
            if last_node == goal or last_node not in graph:
                continue

            for neighbor, weight in graph[last_node]:
                if neighbor not in path:  # Avoid simple cycles
                    new_path = path + [neighbor]
                    new_g = g_val + weight
                    new_f = new_g + heuristics[neighbor]
                    all_candidates.append((new_f, new_g, new_path))

        if not all_candidates:
            print("No more paths to expand.")
            break

        # Sort all candidates by their f(n) value (ascending order)
        all_candidates.sort(key=lambda x: x[0])

        # Prune: keep only the top W paths
        current_beams = all_candidates[:beam_width]
        print(f"Candidates generated: {len(all_candidates)} | Kept top {beam_width}")

    print("\nNo path found to goal.")
    return None, float("inf")


In [2]:
# graph representation: dict of lists of tuples (neighbor, edge_cost)
graph = {
    "A": [("B", 2), ("C", 4)],
    "B": [("D", 3), ("E", 1)],
    "C": [("F", 5), ("G", 2)],
    "D": [("H", 4)],
    "E": [("H", 2), ("I", 6)],
    "F": [("Goal", 1)],
    "G": [("Goal", 3)],
    "H": [("Goal", 2)],
    "I": [("Goal", 1)],
    "Goal": [],
}

# Heuristic values (h) representing estimated cost to Goal
heuristics = {
    "A": 8,
    "B": 6,
    "C": 5,
    "D": 4,
    "E": 3,
    "F": 2,
    "G": 3,
    "H": 2,
    "I": 1,
    "Goal": 0,
}


### Running Beam Search with different Beam Widths

We will run the search with $W = 1$ (greedy selection) and $W = 2$ (retains more paths) to observe the impact on path quality.


In [3]:
# Test with Beam Width = 1
path_w1, cost_w1 = beam_search(
    graph, heuristics, start="A", goal="Goal", beam_width=1
)
print("-" * 50)
print(f"Beam Width 1 Result: Path: {path_w1} | Cost: {cost_w1}")

print("\n" + "=" * 60 + "\n")

# Test with Beam Width = 2
path_w2, cost_w2 = beam_search(
    graph, heuristics, start="A", goal="Goal", beam_width=2
)
print("-" * 50)
print(f"Beam Width 2 Result: Path: {path_w2} | Cost: {cost_w2}")


Starting Beam Search with Beam Width W = 1

--- Step 1 ---
Active Path: A (f: 8, g: 0)
Candidates generated: 2 | Kept top 1
--- Step 2 ---
Active Path: A -> B (f: 8, g: 2)
Candidates generated: 2 | Kept top 1
--- Step 3 ---
Active Path: A -> B -> E (f: 6, g: 3)
Candidates generated: 2 | Kept top 1
--- Step 4 ---
Active Path: A -> B -> E -> H (f: 7, g: 5)
Candidates generated: 1 | Kept top 1
--- Step 5 ---
Active Path: A -> B -> E -> H -> Goal (f: 7, g: 7)

Goal Reached!
--------------------------------------------------
Beam Width 1 Result: Path: ['A', 'B', 'E', 'H', 'Goal'] | Cost: 7


Starting Beam Search with Beam Width W = 2

--- Step 1 ---
Active Path: A (f: 8, g: 0)
Candidates generated: 2 | Kept top 2
--- Step 2 ---
Active Path: A -> B (f: 8, g: 2)
Active Path: A -> C (f: 9, g: 4)
Candidates generated: 4 | Kept top 2
--- Step 3 ---
Active Path: A -> B -> E (f: 6, g: 3)
Active Path: A -> B -> D (f: 9, g: 5)
Candidates generated: 3 | Kept top 2
--- Step 4 ---
Active Path: A -> B -